<a href="https://colab.research.google.com/github/DominicVerschoor/TVA_Strategic-Voting/blob/Happiness/Happiness_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# For one winner

In [ ]:
import numpy as np

def rank_based_happiness(preferences, winner):
    # happiness to be between 0 (least happy, when the winner is the worst choice) and 1 (happiest, when the top preference wins).
    m = len(preferences[0])  # Number of alternatives
    happiness = [(m - voter_prefs.index(winner)) / (m) for voter_prefs in preferences]
    return happiness

In [ ]:
def utility_based_happiness(preferences, winner, utility='linear'):
    # Linear behaves like the rank based happiness but exponential punishes more far away rankings
    m = len(preferences[0])
    happiness = []

    for voter_prefs in preferences:
        rank = voter_prefs.index(winner)
        if utility == 'linear':
            happiness.append((m - rank) / m)
        elif utility == 'exponential':
            happiness.append((2 ** (m - rank) - 1) / (2 ** m - 1))

    return happiness

In [ ]:
def borda_score_happiness(preferences, winner):
    # Like the Borda score method
    m = len(preferences[0])
    happiness = []

    for voter_prefs in preferences:
        borda_scores = {alt: m - voter_prefs.index(alt) - 1 for alt in voter_prefs}
        happiness.append(borda_scores[winner] / max(borda_scores.values()))

    return happiness

In [ ]:
preferences = [['A', 'B', 'C'], ['B', 'C', 'A'], ['C', 'A', 'B']]
winner = 'B'

print("Rank-Based Happiness:", rank_based_happiness(preferences, winner))
print("Utility-Based Happiness (Linear):", utility_based_happiness(preferences, winner, utility='linear'))
print("Utility-Based Happiness (Exponential):", utility_based_happiness(preferences, winner, utility='exponential'))
print("Borda Score Happiness:", borda_score_happiness(preferences, winner))

Rank-Based Happiness: [0.6666666666666666, 1.0, 0.3333333333333333]
Utility-Based Happiness (Linear): [0.6666666666666666, 1.0, 0.3333333333333333]
Utility-Based Happiness (Exponential): [0.42857142857142855, 1.0, 0.14285714285714285]
Borda Score Happiness: [0.5, 1.0, 0.0]


# For position of each candidate

In [ ]:
preferences = [['A', 'B', 'C'], ['B', 'A', 'C'], ['C', 'A', 'B']]
ranking = ['A', 'B', 'C']

#print("Rank-Based Happiness:", rank_based_happiness(preferences, ranking))
#print("Utility-Based Happiness (Linear):", utility_based_happiness(preferences, ranking, utility='linear'))
#print("Utility-Based Happiness (Exponential):", utility_based_happiness(preferences, ranking, utility='exponential'))
print("Borda Score Happiness:", borda_score_happiness_multi(preferences, ranking))

Borda Score Happiness: {'A': 4, 'B': 3, 'C': 2}


In [ ]:
from scipy.stats import kendalltau

def compute_happiness(preferences, final_ranking):
    n = len(final_ranking)  # Number of candidates
    happiness_scores = []
    array = [5,1,1,5]

    for voter in preferences:
        if(voter[0]==final_ranking[0]):
            pos_score = 100 #maximum happiness because we got our top 1
        else:
          # Compute Positional Satisfaction Score
          pos_score = sum(array[i]*(n - abs(voter.index(c) - final_ranking.index(c))) for i, c in enumerate(voter))
          #print(pos_score)
          # Compute Kendall's Tau Distance (Pairwise Agreement)
          tau, _ = kendalltau([voter.index(c) for c in final_ranking], list(range(n)))
          tau_score = (tau + 1) / 2  # Normalize to [0,1]
          #print("tau score")
          #print(tau_score)
          # First and Last Choice Bonus
          first_bonus = 1 if voter[0] == final_ranking[0] else 0  # First choice matches top rank
          last_bonus = 1 if voter[-1] == final_ranking[-1] else 0  # Last choice matches last rank

        # Compute Final Happiness Score
        #happiness = pos_score + tau_score * n + first_bonus + last_bonus
        happiness = pos_score
        happiness_scores.append(happiness)

    return happiness_scores

# Example input
preferences = [['A', 'B', 'C','D'], ['B', 'C', 'A','D'], ['D','C', 'A', 'B']]
winner = ['B', 'A', 'C','D']  # Final ranking

# Compute happiness
happiness_scores = compute_happiness(preferences, winner)
print(happiness_scores)

[42, 100, 16]


In [ ]:
from scipy.stats import kendalltau

def compute_happiness(preferences, final_ranking):
    n = len(final_ranking)  # Number of candidates
    happiness_scores = []
    array = [5,1,1,5] # TODO make it dynamic n on the boundaries and convert to 1 in the middle

    for voter in preferences:
        if(voter[0]==final_ranking[0]):
            pos_score = 100 #TODO maximum happiness because we got our top 1, find the maximum possible value found it, it's each value of the array time n
        else:
          # Compute Positional Satisfaction Score
          pos_score = sum(array[i]*(n - abs(voter.index(c) - final_ranking.index(c))) for i, c in enumerate(voter))

        happiness = pos_score # TODO divide by the maximum number and get from 0 to 1
        happiness_scores.append(happiness)

    return happiness_scores

# Example input
preferences = [['A', 'B', 'C','D'], ['B', 'C', 'A','D'], ['D','C', 'A', 'B']] #Put a dataframe later on (for analyzing)
winner = ['B', 'C', 'A','D'] # Final ranking

# Compute happiness
happiness_scores = compute_happiness(preferences, winner)
print(happiness_scores)

[36, 48, 18]
